# SSCOFS -- checking things out.

This is looking at the raw netcdf files on the NODD.

In [ ]:
from pathlib import Path
import netCDF4
import xarray as xr

import requests


In [ ]:
# sample file.
# auto downloaded from:
base_url = "https://noaa-nos-ofs-pds.s3.amazonaws.com/sscofs/netcdf/2026/11/11/"
infile = Path("sscofs.t03z.20261111.fields.f002.nc")

if (infile).is_file():
    print(infile, "already there")
else:
    print("downloading file")
    response = requests.get(base_url + infile)
    response.raise_for_status()

    with open(infile, "wb") as file:
        file.write(response.content)
    if not (infile).is_file():
        print("Didn't download !!! ")


In [ ]:
ds = xr.open_dataset(infile)

In [ ]:
ds

In [ ]:
print(ds.dims)

for var_name, da in ds.data_vars.items():
    print(f"{var_name}: {da.encoding.get('chunksizes')}")

In [ ]:
print("vars are:", [*ds])


In [ ]:
ds.dims


In [ ]:
433410 / 144470

In [ ]:
ds['u']


In [ ]:
144470 * 4 / 1024 /1024 * 4
# 2.2 MB chunks.

# Try subsetting

In [ ]:
import xarray_subset_grid as xsg

# small
# BBox = (-125.3, 44.29, -123.8, 45.11)
# larger
BBox = (-122, 47.5, -123.5, 48.1)


In [ ]:
ds = xr.open_dataset("sscofs.t03z.20261111.fields.f002.nc")

In [ ]:
ds = xsg.grids.ugrid.assign_ugrid_topology(ds, face_node_connectivity='nv')

In [ ]:
ds2 = ds.xsg.subset_vars(['u', 'v', 'temp', 'salinity'])


In [ ]:
ds3 = ds2.xsg.subset_bbox(BBox)

In [ ]:
ds3.to_netcdf("sscofs_subset.nc")